In [ ]:
# My T4 local Confirmatory run
# I run one model family per checkpoint/session. Change SLOT_TO_RUN to qwen_7b, gemma_9b, llama_8b, or mistral_7b for the next session.
SLOT_TO_RUN = "qwen_7b"
assert SLOT_TO_RUN in {"qwen_7b", "gemma_9b", "llama_8b", "mistral_7b"}
NUM_ITEMS = 260
POOL_SHA256 = "128ff78bb6beb1640fa51897ab97dbb9d07dc56de276973d05a48dda524b53d3"

import hashlib
import json
import os
from pathlib import Path
import shutil
import subprocess
import sys
import zipfile

from google.colab import drive, files

# I keep every completed-item checkpoint in Drive so a runtime reset does not erase it.
drive.mount("/content/drive")
checkpoint_root = Path("/content/drive/MyDrive/DesignLab_v9_local_hf_confirmatory")
checkpoint_root.mkdir(parents=True, exist_ok=True)

# I install only local inference libraries. These downloads are model weights and software, not inference API calls.
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "--quiet",
    "transformers>=4.48,<5", "accelerate>=1.2,<2", "bitsandbytes>=0.45,<1",
    "safetensors>=0.4", "sentencepiece>=0.2,<1",
])

# I do not load or use an OpenRouter, Together, Groq, Gemini, or Hugging Face API key.
os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
os.environ["HF_HOME"] = "/content/hf_cache"

uploaded = files.upload()
bundle_names = [name for name in uploaded if name.lower().endswith("v9_0_colab_t4_local_confirmatory_bundle.zip")]
if len(bundle_names) != 1:
    raise RuntimeError("Upload exactly v9_0_colab_t4_local_confirmatory_bundle.zip.")
bundle_path = Path("/content") / bundle_names[0]
unpack_root = Path("/content/designlab_local_hf_bundle")
if unpack_root.exists():
    shutil.rmtree(unpack_root)
unpack_root.mkdir(parents=True)
with zipfile.ZipFile(bundle_path) as archive:
    bad_member = archive.testzip()
    if bad_member:
        raise RuntimeError("The uploaded bundle failed its ZIP integrity check.")
    for member in archive.infolist():
        parts = Path(member.filename).parts
        if Path(member.filename).is_absolute() or ".." in parts:
            raise RuntimeError("The uploaded bundle contains an unsafe archive path.")
    archive.extractall(unpack_root)

research_dir = unpack_root / "research" / "v9_0_prospective"
pool_path = unpack_root / "items_pool_v1.json"
manifest_path = research_dir / "MANIFEST_SHA256.txt"
if not research_dir.is_dir() or not pool_path.is_file() or not manifest_path.is_file():
    raise RuntimeError("The bundle is missing its v9 runner, manifest, or item pool.")
for line in manifest_path.read_text(encoding="utf-8").splitlines():
    digest, relative_name = line.split("  ", 1)
    artifact = research_dir / relative_name
    actual = hashlib.sha256(artifact.read_bytes()).hexdigest()
    if actual != digest:
        raise RuntimeError("Manifest verification failed for " + relative_name)
pool_hash = hashlib.sha256(pool_path.read_bytes()).hexdigest()
if pool_path.stat().st_size != 75875 or pool_hash != POOL_SHA256:
    raise RuntimeError("The frozen 260-item pool size or SHA-256 does not match.")

import torch
if not torch.cuda.is_available():
    raise RuntimeError("Select Runtime type: T4 GPU before running this notebook.")
gpu_name = torch.cuda.get_device_name(0)
if "T4" not in gpu_name.upper():
    raise RuntimeError("This notebook is locked to a Colab T4 GPU; detected: " + gpu_name)
print("Verified 260-item pool and package manifest. GPU:", gpu_name)
print("Local model API spend: USD 0.00. Colab free-tier GPU availability and runtime limits are controlled by Colab.")

sys.path.insert(0, str(research_dir))
script_path = research_dir / "confirmatory_v9.py"
checkpoint_path = checkpoint_root / ("confirmatory_local_hf_" + SLOT_TO_RUN + ".json")
command = [
    sys.executable, str(script_path),
    "--backend", "local-hf",
    "--slot", SLOT_TO_RUN,
    "--items", str(pool_path),
    "--output", str(checkpoint_path),
    "--execute",
]
if checkpoint_path.exists():
    command.extend(["--resume-from", str(checkpoint_path)])
print("Running slot:", SLOT_TO_RUN, "with checkpoint:", checkpoint_path)
completed = subprocess.run(command, cwd=research_dir, text=True, check=False)
if checkpoint_path.exists():
    saved = json.loads(checkpoint_path.read_text(encoding="utf-8"))
    summary = saved.get("per_model_summary", [{}])[0]
    print("Saved status:", saved.get("run_status"), "completed items:", saved.get("completed_items"))
    print("Local generation calls:", saved.get("local_generation_calls"), "model API spend: USD 0.00")
else:
    print("No checkpoint was created; inspect the runner output above.")
print("Runner exit code:", completed.returncode)

# I merge only once all four matching per-slot checkpoints exist; partial checkpoints remain usable on their own.
slot_names = ["qwen_7b", "gemma_9b", "llama_8b", "mistral_7b"]
slot_paths = [checkpoint_root / ("confirmatory_local_hf_" + slot + ".json") for slot in slot_names]
if all(path.is_file() for path in slot_paths):
    from local_hf_confirmatory_v9 import merge_slot_checkpoints
    merged_path = checkpoint_root / "merged_confirmatory_local_hf_v9.json"
    try:
        merged = merge_slot_checkpoints(slot_paths, merged_path)
        print("Merged four-family result:", merged_path)
        print("All four 260-item pipelines complete:", merged["complete_pipeline"])
    except (ValueError, KeyError, json.JSONDecodeError) as exc:
        print("Merge is waiting for four compatible checkpoints:", str(exc))
else:
    missing = [slot for slot, path in zip(slot_names, slot_paths) if not path.is_file()]
    print("Run the remaining model slots in separate sessions:", ", ".join(missing))
